In [1]:
# Debug & robust entity-type inference for DRKG processed_graph
import os, json, sys, math, re
from collections import Counter, defaultdict
import torch
import pandas as pd

GRAPH_DIR = "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph" 
print("GRAPH_DIR =", GRAPH_DIR)

# file paths
edge_index_path = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/edge_index.pt")
edge_type_path = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/edge_type.pt")
entities_path = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/entities.txt")
relations_path = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/relations.txt")

train_csv = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/train_inductive.csv")
val_csv = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/val_inductive.csv")
test_csv = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/test_inductive.csv")

def safe_read_lines(path, n=50):
    if not os.path.exists(path):
        print(f"  (missing) {path}")
        return []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = [ln.rstrip("\n") for ln in f]
    print(f"Loaded {len(lines)} lines from {path}")
    return lines

# 1) Show samples of files
entities_lines = safe_read_lines(entities_path, 50)
relations_lines = safe_read_lines(relations_path, 200)
print("\n-- sample entities (first 30) --")
for ln in entities_lines[:30]:
    print(ln)
print("\n-- sample relations (first 50) --")
for ln in relations_lines[:50]:
    print(ln)

# 2) Load edge_index / edge_type if available
if os.path.exists(edge_index_path) and os.path.exists(edge_type_path):
    edge_index = torch.load(edge_index_path)
    edge_type = torch.load(edge_type_path)
    print("\nLoaded edge_index shape:", tuple(edge_index.shape))
    print("Loaded edge_type length:", len(edge_type))
else:
    edge_index = None; edge_type = None
    print("\nedge_index.pt or edge_type.pt missing")

# 3) Load inductive CSV triples (if present)
def read_csv_triples(path):
    if not os.path.exists(path):
        return []
    try:
        df = pd.read_csv(path, header=None, dtype=str)
        if df.shape[1] >= 3:
            triples = [(str(r[0]).strip(), str(r[1]).strip(), str(r[2]).strip()) for r in df.values]
        else:
            triples = []
        return triples
    except Exception as e:
        print("Failed to read csv:", path, "err:", e)
        return []

train_triples = read_csv_triples(train_csv)
val_triples = read_csv_triples(val_csv)
test_triples = read_csv_triples(test_csv)
print(f"\ntrain/val/test triples counts: {len(train_triples)}/{len(val_triples)}/{len(test_triples)}")
print("sample train triples (first 30):")
for t in train_triples[:30]:
    print(t)

# 4) Try multiple entity->gid mappings and heuristics
# Option A: entities.txt lists one entity id per line -> map index -> line
ent_map_by_line = {}
for i,ln in enumerate(entities_lines):
    key = ln.strip()
    ent_map_by_line[key] = i  # mapping from entity string to gid (line-index)
# Option B: if entities.txt contains tab-separated fields like "id \t name \t type", parse id
ent_map_by_field = {}
for i, ln in enumerate(entities_lines):
    if "\t" in ln:
        parts = ln.split("\t")
        # common cases: id \t name \t type OR id \t name OR id \t label
        ent_id = parts[0].strip()
        ent_map_by_field[ent_id] = i
    elif "," in ln:
        parts = ln.split(",")
        ent_id = parts[0].strip()
        ent_map_by_field[ent_id] = i
# Option C: if entities are stored with an explicit index like "123\t<name>", support 'index' match
ent_map_index_str = {}
for i, ln in enumerate(entities_lines):
    # if the line begins with an integer id
    m = re.match(r"^\s*(\d+)\s*[\t, ]", ln)
    if m:
        ent_map_index_str[m.group(1)] = i

print("\nMap sizes: by_line={}, by_field={}, index_str={}".format(len(ent_map_by_line),
                                                                 len(ent_map_by_field),
                                                                 len(ent_map_index_str)))
# print example keys to inspect
def head_keys(d, k=10):
    return list(d.keys())[:k]

print("example entity keys (by_line):", head_keys(ent_map_by_line))
print("example entity keys (by_field):", head_keys(ent_map_by_field))
print("example index-string keys:", head_keys(ent_map_index_str))

# 5) Entity type heuristics: direct column, keyword in entity string, and relation-based inference
def infer_type_from_entity_line(ln):
    ln_low = ln.lower()
    if "\t" in ln:
        parts = ln.split("\t")
        if len(parts) >= 3:
            typ = parts[-1].strip().lower()
            if typ in ("compound","drug","gene","disease","other"):
                if typ == "drug": return "compound"
                return typ
    # check common tokens
    if any(x in ln_low for x in ["drug","compound","db","cid:","chem"]): return "compound"
    if any(x in ln_low for x in ["gene","hgnc:","entrez","symbol","ensp","ensg"]): return "gene"
    if any(x in ln_low for x in ["disease","doid","mesh","omim","phenotype"]): return "disease"
    # default
    return "other"

# try build entity-type by parsing entities_lines
ent_type_guess = {}
for i, ln in enumerate(entities_lines):
    ent_type_guess[i] = infer_type_from_entity_line(ln)

print("\nType guess counts from entity lines:", Counter(ent_type_guess.values()))

# 6) Relation name mapping (if relations.txt exists)
rel_map = {}
if len(relations_lines) > 0:
    # relations.txt may have one relation name per line; otherwise parse first token
    for i, ln in enumerate(relations_lines):
        name = ln.strip().split("\t")[0].strip()
        rel_map[i] = name
    print("Loaded relations (example):", list(rel_map.items())[:20])
else:
    print("relations.txt not found / empty")

# 7) Relation-name based inference: for nodes that frequently appear on one side of certain relation names,
#    infer their type by transfer. For example, if node X appears as tail in many relations whose name contains 'gene',
#    label X as gene.
node_relation_counts = defaultdict(lambda: defaultdict(int))  # node_gid -> relname -> count
# if train_triples present and ent_map_by_line likely maps string->gid, attempt mapping
all_triplets = train_triples + val_triples + test_triples
mapped_any = 0
for (h,r,t) in all_triplets:
    mapped = None
    # try multiple maps
    if h in ent_map_by_field and t in ent_map_by_field:
        hid, tid = ent_map_by_field[h], ent_map_by_field[t]; mapped_any += 1
    elif h in ent_map_by_line and t in ent_map_by_line:
        hid, tid = ent_map_by_line[h], ent_map_by_line[t]; mapped_any += 1
    elif h in ent_map_index_str and t in ent_map_index_str:
        hid, tid = ent_map_index_str[h], ent_map_index_str[t]; mapped_any += 1
    else:
        # try numeric coercion
        try:
            hi = int(h); ti = int(t)
            if 0 <= hi < len(entities_lines) and 0 <= ti < len(entities_lines):
                hid, tid = hi, ti; mapped_any += 1
            else:
                continue
        except:
            # skip if cannot map
            continue
    # accumulate relation-name based counts
    relname = str(r).lower()
    node_relation_counts[hid][relname] += 1
    node_relation_counts[tid][relname] += 1

print(f"\nMapped {mapped_any} out of {len(all_triplets)} triples to entity indices via heuristics")

# Build relation-signature keywords mapping (adjust if your relation names include other tokens)
gene_tokens = ["gene","target","hgnc","entrez","transcript","expression"]
drug_tokens = ["drug","compound","compound","therapeutic","drugbank","db"]
disease_tokens = ["disease","disorder","phenotype","doid","mesh","omim","indication","treat"]

# infer node type from relation co-occurrence
rel_based_type = {}
for nid, rel_counts in node_relation_counts.items():
    # score for each class by counting rel names that contain any token
    scores = {"gene":0, "compound":0, "disease":0}
    for relname, cnt in rel_counts.items():
        rn = relname.lower()
        if any(tok in rn for tok in gene_tokens):
            scores["gene"] += cnt
        if any(tok in rn for tok in drug_tokens):
            scores["compound"] += cnt
        if any(tok in rn for tok in disease_tokens):
            scores["disease"] += cnt
    # if all zero, skip
    if sum(scores.values())==0:
        continue
    # choose best
    typ = max(scores.items(), key=lambda x: x[1])[0]
    rel_based_type[nid] = typ

print("Relation-based type inferred counts:", Counter(rel_based_type.values()))

# 8) Combine heuristics: prefer explicit column parse > entity-string-heuristic > relation-based > fallback 'other'
final_type = {}
for idx in range(len(entities_lines)):
    if idx in ent_type_guess and ent_type_guess[idx] in ("compound","gene","disease"):
        final_type[idx] = ent_type_guess[idx]
    elif idx in rel_based_type:
        final_type[idx] = rel_based_type[idx]
    else:
        final_type[idx] = "other"

print("Final type counts:", Counter(final_type.values()))
print("\nSample nodes by type (up to 10 each):")
for t in ("compound","gene","disease","other"):
    examples = [ (i, entities_lines[i]) for i,ty in final_type.items() if ty==t ]
    print(f"\nType {t} (count={len(examples)}), examples:")
    for ex in examples[:10]:
        print("  ", ex)

# 9) Now attempt to extract mechanistic pairs using final_type + relation-name heuristics
comp_gene = set(); gene_disease = set(); comp_disease = set()
for (h,r,t) in all_triplets:
    mapped = None
    # mapping heuristics (same as above)
    if h in ent_map_by_field and t in ent_map_by_field:
        hid, tid = ent_map_by_field[h], ent_map_by_field[t]
    elif h in ent_map_by_line and t in ent_map_by_line:
        hid, tid = ent_map_by_line[h], ent_map_by_line[t]
    elif h in ent_map_index_str and t in ent_map_index_str:
        hid, tid = ent_map_index_str[h], ent_map_index_str[t]
    else:
        try:
            hi=int(h); ti=int(t)
            hid, tid = hi, ti
        except:
            continue
    rl = str(r).lower()
    # direct type-based
    if final_type.get(hid,"other")=="compound" and final_type.get(tid,"other")=="gene":
        comp_gene.add((hid, tid))
    if final_type.get(hid,"other")=="gene" and final_type.get(tid,"other")=="disease":
        gene_disease.add((hid, tid))
    if final_type.get(hid,"other")=="compound" and final_type.get(tid,"other")=="disease":
        comp_disease.add((hid, tid))
    # relation based matching
    if any(tok in rl for tok in ["target","bind","interact"]) and final_type.get(hid,"other")=="compound":
        comp_gene.add((hid, tid))
    if any(tok in rl for tok in ["associate","assoc","gene_assoc","gene_association"]) and final_type.get(tid,"other")=="disease":
        gene_disease.add((hid, tid))
    if any(tok in rl for tok in ["treat","indicat","therap"]) and final_type.get(hid,"other")=="compound":
        comp_disease.add((hid, tid))

print("\nExtracted mechanistic counts (comp->gene, gene->disease, comp->disease):", len(comp_gene), len(gene_disease), len(comp_disease))

# 10) Diagnostic: show some triples that could not be mapped or that caused 0 earlier
unmapped_triples = []
for (h,r,t) in train_triples[:200]:
    ok = False
    # can we map?
    if h in ent_map_by_field or h in ent_map_by_line or h in ent_map_index_str:
        ok=True
    else:
        try:
            int(h); ok=True
        except:
            ok=False
    if not ok:
        unmapped_triples.append((h,r,t))
print("\nExamples of unmapped train triples (first 30):")
for ut in unmapped_triples[:30]:
    print("  ", ut)

# 11) Save diagnostic JSON for you to paste back
diag = {
    "num_entities_lines": len(entities_lines),
    "num_relations_lines": len(relations_lines),
    "num_train_triples": len(train_triples),
    "entity_map_by_line_sample": list(head_keys := list(ent_map_by_line.keys())[:20]),
    "entity_map_by_field_sample": list(list(ent_map_by_field.keys())[:20]),
    "entity_index_str_sample": list(list(ent_map_index_str.keys())[:20]),
    "final_type_counts": dict(Counter(final_type.values())),
    "sample_comp_gene": list(list(comp_gene)[:20]),
    "sample_gene_dis": list(list(gene_disease)[:20]),
    "sample_comp_dis": list(list(comp_disease)[:20]),
    "unmapped_train_examples": unmapped_triples[:50]
}
out_diag = os.path.join(GRAPH_DIR, "diagnostic_entity_parsing.json")
with open(out_diag, "w") as f:
    json.dump(diag, f, indent=2)
print("\nWrote diagnostic file to", out_diag)

print("\nIf comp->gene or gene->disease counts are still zero, please paste the following outputs here or attach them:")
print("  - first 30 lines of entities.txt (printed above)")
print("  - first 50 relations.txt lines (printed above)")
print("  - first 30 train_inductive.csv triples (printed above)")

# End of cell


GRAPH_DIR = C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph
Loaded 94046 lines from C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/entities.txt
Loaded 107 lines from C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/relations.txt

-- sample entities (first 30) --
0	Anatomy::UBERON:0000002
1	Anatomy::UBERON:0000004
2	Anatomy::UBERON:0000006
3	Anatomy::UBERON:0000007
4	Anatomy::UBERON:0000010
5	Anatomy::UBERON:0000011
6	Anatomy::UBERON:0000013
7	Anatomy::UBERON:0000020
8	Anatomy::UBERON:0000026
9	Anatomy::UBERON:0000029
10	Anatomy::UBERON:0000033
11	Anatomy::UBERON:0000038
12	Anatomy::UBERON:0000042
13	Anatomy::UBERON:0000043
14	Anatomy::UBERON:0000045
15	Anatomy::UBERON:0000053
16	Anatomy::UBERON:0000054
17	Anatomy::UBERON:0000056
18	Anatomy::UBERON:0000057
19	Anatomy::UBERON:0000165
20	Anatomy::UBERON:0000178
21	Anatomy::UBERON:0000211
22	Anatom